# Humvee detector: scratch vs. Carparts initialization

This notebook creates one deterministic 70/15/15 group-aware split and trains the same YOLOX-S model twice. Only initialization changes. Run all cells with the `Python 3.10 (crater)` kernel.

> This uses an interim group-aware split based on Commons upload sequences and visually confirmed links. Authoritative provenance metadata is still required before treating results as a release benchmark.

## 1. Setup

The parent checkpoint's SHA-256 is its file fingerprint. Checking it confirms that the local `.pth` file is exactly the promoted Carparts model, rather than another checkpoint with the same filename.

In [ ]:
import hashlib
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import torch

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'third_party' / 'YOLOX').is_dir())
SOURCE_DIR = ROOT / 'datasets' / 'military' / 'components' / 'source' / 'humvee'
SOURCE_JSON = SOURCE_DIR / 'instances_default.json'
ANNOTATION_DIR = SOURCE_DIR.parents[1] / 'annotations'
PARENT_CHECKPOINT = ROOT / 'outputs' / 'detection' / 'civilian' / 'promoted' / 'carparts23-yolox-s-coco-v1' / 'carparts23-yolox-s-coco-v1.pth'
PROMOTED_PARENT_SHA256 = '5ab8c99a5a67ca253aea6f1d4a6974f827b5e5bd16aca227306c129a7fd94d47'

assert torch.cuda.is_available(), 'Use the CUDA-enabled crater environment.'
assert SOURCE_JSON.is_file(), f'Missing source annotations: {SOURCE_JSON}'
assert PARENT_CHECKPOINT.is_file(), f'Missing promoted checkpoint: {PARENT_CHECKPOINT}'
assert hashlib.sha256(PARENT_CHECKPOINT.read_bytes()).hexdigest() == PROMOTED_PARENT_SHA256, 'The parent checkpoint is not the promoted model.'
print(f'PyTorch {torch.__version__}; {torch.cuda.get_device_name(0)}')

## 2. Prepare a leakage-resistant dataset split

Related Commons upload sequences are assigned as groups, then each group is placed wholly in train, validation, or test. The source export remains unchanged.

In [ ]:
import re

payload = json.loads(SOURCE_JSON.read_text(encoding='utf-8'))
source_id = lambda image: int(re.search(r'\d+', image['file_name']).group())
images = sorted(payload['images'], key=source_id)
assert len(payload['categories']) == 6, 'Expected six source categories.'
assert all((SOURCE_DIR / image['file_name']).is_file() for image in images), 'A source image is missing.'

MAX_SERIES_ID_GAP = 5_000
photo_groups = []
for image in images:
    if not photo_groups or source_id(image) - source_id(photo_groups[-1][-1]) > MAX_SERIES_ID_GAP:
        photo_groups.append([])
    photo_groups[-1].append(image)

VISUALLY_LINKED_SERIES = [
    {'commons_72095749.jpg', 'commons_73531531.jpg'},
    {'commons_52358471.jpg', 'commons_53183995.jpg', 'commons_53326023.jpg'},
    {'commons_40286022.jpg', 'commons_56467136.jpg'},
]
for related_names in VISUALLY_LINKED_SERIES:
    related_groups = [group for group in photo_groups if related_names.intersection(image['file_name'] for image in group)]
    photo_groups = [group for group in photo_groups if group not in related_groups] + [sum(related_groups, [])]

random.Random(42).shuffle(photo_groups)
photo_groups.sort(key=len, reverse=True)
targets = {'train': 0.70 * len(images), 'val': 0.15 * len(images), 'test': 0.15 * len(images)}
split_images = {split: [] for split in targets}
for group in photo_groups:
    split = max(targets, key=lambda name: (targets[name] - len(split_images[name])) / targets[name])
    split_images[split].extend(group)

group_by_image = {int(image['id']): group for group, items in enumerate(photo_groups) for image in items}
group_split = {}
for split, selected_images in split_images.items():
    for image in selected_images:
        group = group_by_image[int(image['id'])]
        assert group_split.setdefault(group, split) == split, 'A photo series crosses splits.'

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
for split, selected_images in split_images.items():
    selected_images = sorted(selected_images, key=source_id)
    image_ids = {int(image['id']) for image in selected_images}
    annotations = [item for item in payload['annotations'] if int(item['image_id']) in image_ids]
    assert len({int(item['category_id']) for item in annotations}) == 6, f'{split} is missing a category.'
    output_payload = {
        'info': payload.get('info', {}),
        'licenses': payload.get('licenses', []),
        'categories': payload['categories'],
        'images': [{**image, 'file_name': f"source/humvee/{image['file_name']}"} for image in selected_images],
        'annotations': annotations,
    }
    output = ANNOTATION_DIR / f'humvee_source6_instances_{split}.json'
    output.write_text(json.dumps(output_payload, indent=2) + '\n', encoding='utf-8')
    print(f'{split}: {len(selected_images)} images, {len(annotations)} boxes')
print(f'{len(photo_groups)} independent photo groups; largest group: {max(map(len, photo_groups))} images; cross-split groups: 0')

## 3. Inspect validation and test images

Validation and test each render as one annotated grid, so every image is visible at once.

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

def inspect_split(split, columns=5, tile_size=(260, 210)):
    data = json.loads((ANNOTATION_DIR / f'humvee_source6_instances_{split}.json').read_text(encoding='utf-8'))
    class_names = {int(item['id']): item['name'] for item in data['categories']}
    boxes_by_image = {}
    for annotation in data['annotations']:
        boxes_by_image.setdefault(int(annotation['image_id']), []).append(annotation)

    rows = (len(data['images']) + columns - 1) // columns
    grid = Image.new('RGB', (columns * tile_size[0], rows * tile_size[1]), 'white')
    grid_draw = ImageDraw.Draw(grid)
    for index, image_record in enumerate(data['images']):
        with Image.open(ANNOTATION_DIR.parent / image_record['file_name']) as source:
            image = source.convert('RGB')
        image.thumbnail((tile_size[0] - 10, tile_size[1] - 35))
        scale_x = image.width / image_record['width']
        scale_y = image.height / image_record['height']
        draw = ImageDraw.Draw(image)
        for annotation in boxes_by_image.get(int(image_record['id']), []):
            x, y, width, height = annotation['bbox']
            box = (x * scale_x, y * scale_y, (x + width) * scale_x, (y + height) * scale_y)
            label = class_names[int(annotation['category_id'])]
            draw.rectangle(box, outline='lime', width=3)
            draw.text((box[0], box[1]), label, fill='lime', stroke_width=2, stroke_fill='black')
        column, row = index % columns, index // columns
        left = column * tile_size[0] + (tile_size[0] - image.width) // 2
        top = row * tile_size[1] + 25
        grid.paste(image, (left, top))
        name = Path(image_record['file_name']).name
        grid_draw.text((column * tile_size[0] + 5, row * tile_size[1] + 5), name, fill='black')
    display(grid)

In [ ]:
inspect_split('val')

In [ ]:
inspect_split('test')

## 4. Configure training

Both runs use 100 epochs, batch size 8, one GPU, mixed precision, seed 42, and the same prepared annotations. The progress bar shows epoch/iteration progress, loss, learning rate, GPU memory, and validation AP. Full loss and validation histories are also written to TensorBoard.

In [ ]:
import re
import time
from tqdm.auto import tqdm

TRAIN_SCRIPT = ROOT / 'third_party' / 'YOLOX' / 'tools' / 'train.py'
EXP_FILE = ROOT / 'experiments' / 'detection' / 'yolox_s_crater6.py'
OUTPUT_ROOT = ROOT / 'outputs' / 'detection' / 'humvee_source6'
COMMON_ARGS = [sys.executable, '-u', str(TRAIN_SCRIPT), '-f', str(EXP_FILE), '-d', '1', '-b', '8', '--fp16', '-l', 'tensorboard']
EXPERIMENT_OVERRIDES = [
    'train_ann', 'humvee_source6_instances_train.json',
    'val_ann', 'humvee_source6_instances_val.json',
    'test_ann', 'humvee_source6_instances_test.json',
    'output_dir', str(OUTPUT_ROOT), 'max_epoch', '100', 'warmup_epochs', '5',
    'no_aug_epochs', '15', 'eval_interval', '5', 'data_num_workers', '4',
]
TRAINING_ENV = {**os.environ, 'TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD': '1'}
ANSI_ESCAPE = re.compile(r'\x1b\[[0-9;]*m')
PROGRESS_PATTERN = re.compile(r'epoch: (\d+)/(\d+), iter: (\d+)/(\d+)')
AP_PATTERN = re.compile(r'Average Precision.*IoU=0.50:0.95.*= ([0-9.]+)')

def train(run_name, checkpoint=None):
    checkpoint_args = ['-c', str(checkpoint)] if checkpoint else []
    command = COMMON_ARGS + ['-expn', run_name] + checkpoint_args + EXPERIMENT_OVERRIDES
    started = time.perf_counter()
    progress = tqdm(total=100, desc=run_name, unit='epoch')
    recent_lines = []
    process = subprocess.Popen(command, cwd=ROOT, env=TRAINING_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for raw_line in process.stdout:
        line = ANSI_ESCAPE.sub('', raw_line).strip()
        recent_lines = (recent_lines + [line])[-20:]
        match = PROGRESS_PATTERN.search(line)
        if match:
            epoch, max_epoch, iteration, max_iteration = map(int, match.groups())
            position = epoch - 1 + iteration / max_iteration
            progress.update(max(0, position - progress.n))
            details = {'iter': f'{iteration}/{max_iteration}'}
            for label, pattern in {'loss': r'total_loss: ([0-9.]+)', 'iou': r'iou_loss: ([0-9.]+)', 'conf': r'conf_loss: ([0-9.]+)', 'lr': r'lr: ([0-9.eE+-]+)', 'gpu_mb': r'gpu mem: ([0-9.]+)Mb'}.items():
                value = re.search(pattern, line)
                if value:
                    details[label] = value.group(1)
            progress.set_postfix(details, refresh=False)
        ap_match = AP_PATTERN.search(line)
        if ap_match:
            tqdm.write(f'{run_name} validation AP50:95: {float(ap_match.group(1)):.3f}')
    return_code = process.wait()
    progress.close()
    if return_code:
        print('\n'.join(recent_lines))
        raise subprocess.CalledProcessError(return_code, command)

    checkpoint_path = OUTPUT_ROOT / run_name / 'best_ckpt.pth'
    result = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    summary = {'run': run_name, 'best_ap50_95': float(result['best_ap']), 'best_epoch': int(result['start_epoch']), 'minutes': (time.perf_counter() - started) / 60}
    print(f"Finished {run_name}: best AP50:95={summary['best_ap50_95']:.4f} at epoch {summary['best_epoch']} in {summary['minutes']:.1f} minutes")
    print(f'TensorBoard: {OUTPUT_ROOT / run_name / "tensorboard"}')
    return summary

## 5. Train from scratch

This run starts with random model weights.

In [ ]:
scratch_summary = train('scratch_seed42')

## 6. Train from the promoted Carparts model

YOLOX loads the compatible backbone, neck, and head tensors. The incompatible 23-class prediction tensors are replaced by the six-class head.

In [ ]:
initialized_summary = train('carparts_initialized_seed42', PARENT_CHECKPOINT)

## 7. Compare validation AP

Select using validation AP only. Keep the test split untouched until the initialization choice is final.

In [ ]:
results = {}
for run_name in ('scratch_seed42', 'carparts_initialized_seed42'):
    result = torch.load(OUTPUT_ROOT / run_name / 'best_ckpt.pth', map_location='cpu', weights_only=False)
    results[run_name] = {'ap': float(result['best_ap']), 'epoch': int(result['start_epoch'])}

print('run | best epoch | validation AP50:95')
print('--- | ---: | ---:')
for run_name, result in results.items():
    print(f"{run_name} | {result['epoch']} | {result['ap']:.4f}")
delta = results['carparts_initialized_seed42']['ap'] - results['scratch_seed42']['ap']
print(f'Carparts initialization delta: {delta:+.4f} AP50:95')

In [ ]:
# 8. Run the selected model on three held-out test images (standalone)
import json
import sys
from pathlib import Path

import cv2
import torch
from IPython.display import display
from PIL import Image, ImageDraw

# Configuration: this cell only requires the saved checkpoint and test annotations.
INFERENCE_RUN = 'carparts_initialized_seed42'
VIS_CONFIDENCE = 0.25
NUM_TEST_IMAGES = 3

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'third_party' / 'YOLOX').is_dir()),
    None,
)
assert ROOT is not None, 'Run this notebook from the CRATER repository or one of its subdirectories.'
YOLOX_ROOT = ROOT / 'third_party' / 'YOLOX'
EXP_FILE = ROOT / 'experiments' / 'detection' / 'yolox_s_crater6.py'
ANNOTATION_FILE = (
    ROOT
    / 'datasets'
    / 'military'
    / 'components'
    / 'annotations'
    / 'humvee_source6_instances_test.json'
)
CHECKPOINT_FILE = (
    ROOT
    / 'outputs'
    / 'detection'
    / 'humvee_source6'
    / INFERENCE_RUN
    / 'best_ckpt.pth'
)
assert EXP_FILE.is_file(), f'Missing experiment definition: {EXP_FILE}'
assert ANNOTATION_FILE.is_file(), f'Missing test annotations: {ANNOTATION_FILE}'
assert CHECKPOINT_FILE.is_file(), f'Missing trained checkpoint: {CHECKPOINT_FILE}'

if str(YOLOX_ROOT) not in sys.path:
    sys.path.insert(0, str(YOLOX_ROOT))

from yolox.data import ValTransform
from yolox.exp import get_exp
from yolox.utils import postprocess

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inference_exp = get_exp(str(EXP_FILE), None)
inference_model = inference_exp.get_model()
checkpoint = torch.load(CHECKPOINT_FILE, map_location='cpu', weights_only=False)
inference_model.load_state_dict(checkpoint['model'])
inference_model.to(device).eval()
preprocess = ValTransform(legacy=False)

test_payload = json.loads(ANNOTATION_FILE.read_text(encoding='utf-8'))
class_names = [
    item['name']
    for item in sorted(test_payload['categories'], key=lambda item: int(item['id']))
]
test_images = sorted(test_payload['images'], key=lambda item: item['file_name'])[:NUM_TEST_IMAGES]
assert len(test_images) == NUM_TEST_IMAGES, f'Expected at least {NUM_TEST_IMAGES} test images.'
image_root = ANNOTATION_FILE.parent.parent

for image_record in test_images:
    image_path = image_root / image_record['file_name']
    raw_bgr = cv2.imread(str(image_path))
    assert raw_bgr is not None, f'Could not read {image_path}'
    ratio = min(
        inference_exp.test_size[0] / raw_bgr.shape[0],
        inference_exp.test_size[1] / raw_bgr.shape[1],
    )
    network_input, _ = preprocess(raw_bgr, None, inference_exp.test_size)
    network_input = torch.from_numpy(network_input).unsqueeze(0).float().to(device)

    with torch.inference_mode():
        detections = postprocess(
            inference_model(network_input),
            inference_exp.num_classes,
            VIS_CONFIDENCE,
            inference_exp.nmsthre,
        )[0]

    rendered = Image.fromarray(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(rendered)
    detection_count = 0 if detections is None else len(detections)
    if detections is not None:
        detections = detections.detach().cpu()
        boxes = detections[:, :4] / ratio
        scores = detections[:, 4] * detections[:, 5]
        class_ids = detections[:, 6].to(torch.int64)
        line_width = max(2, round(min(rendered.size) / 250))
        for box, score, class_id in zip(boxes, scores, class_ids):
            x1, y1, x2, y2 = box.tolist()
            x1, x2 = sorted((max(0, x1), min(rendered.width - 1, x2)))
            y1, y2 = sorted((max(0, y1), min(rendered.height - 1, y2)))
            label = f'{class_names[int(class_id)]} {float(score):.2f}'
            draw.rectangle((x1, y1, x2, y2), outline='lime', width=line_width)
            draw.text(
                (x1, max(0, y1 - 14)),
                label,
                fill='lime',
                stroke_width=2,
                stroke_fill='black',
            )

    rendered.thumbnail((1000, 700))
    print(
        f"{Path(image_record['file_name']).name}: {detection_count} detections "
        f"at confidence >= {VIS_CONFIDENCE:.2f} ({device})"
    )
    display(rendered)